In [ ]:
import csv
import math
from collections import Counter
import random

# =====================================================================
# PASSO 1: Métricas de Proximidade
# =====================================================================
def calcular_distancia_manhattan(ponto_a, ponto_b):
    
    soma_diferencas = 0
    for i in range(len(ponto_a)):
        soma_diferencas += math.fabs(ponto_a[i] - ponto_b[i])
    return soma_diferencas

def calcular_distancia_euclidiana(ponto_a, ponto_b):
    """Calcula a menor linha reta entre dois pontos (métrica tradicional)."""
    soma_quadrados = 0
    for i in range(len(ponto_a)):
        soma_quadrados += (ponto_a[i] - ponto_b[i]) ** 2
    return math.sqrt(soma_quadrados)


# =====================================================================
# PASSO 2: Tratamento de Dados e Normalização
# =====================================================================
def carregar_dados_csv(caminho_arquivo):
    """Abre o CSV e divide os dados entre Atributos (X) e Classes (y)."""
    X_atributos = []
    y_classes = []
    
    with open(caminho_arquivo, mode='r', encoding='utf-8') as arquivo:
        leitor_csv = csv.reader(arquivo)
        header = next(leitor_csv) 
        
        for linha in leitor_csv:
            if not linha:
                continue
            
            atributos = [float(valor) for valor in linha[:-1]]
            classe = linha[-1]
            
            X_atributos.append(atributos)
            y_classes.append(classe)
            
    return X_atributos, y_classes

def normalizar_dados(X):
    
    num_colunas = len(X[0])
    num_linhas = len(X)
    
    
    X_normalizado = [[0.0] * num_colunas for _ in range(num_linhas)]
    
    for col in range(num_colunas):
        
        valores_coluna = [X[linha][col] for linha in range(num_linhas)]
        val_min = min(valores_coluna)
        val_max = max(valores_coluna)
        
        
        for linha in range(num_linhas):
            valor_original = X[linha][col]
            divisao = val_max - val_min
            
            
            if divisao == 0:
                X_normalizado[linha][col] = 0.0
            else:
                X_normalizado[linha][col] = (valor_original - val_min) / divisao
                
    return X_normalizado


# =====================================================================
# PASSO 3: O Algoritmo KNN
# =====================================================================
def classificar_knn(X_treino, y_treino, novo_ponto, k=3, metrica='manhattan'):
    lista_distancias = []
    
    for i in range(len(X_treino)):
        
        if metrica == 'manhattan':
            dist = calcular_distancia_manhattan(X_treino[i], novo_ponto)
        else:
            dist = calcular_distancia_euclidiana(X_treino[i], novo_ponto)
            
        lista_distancias.append((dist, y_treino[i]))
        
    
    lista_distancias.sort(key=lambda item: item[0])
    
    
    k_vizinhos = lista_distancias[:k]
    classes_dos_vizinhos = [vizinho[1] for vizinho in k_vizinhos]
    votacao = Counter(classes_dos_vizinhos).most_common(1)
    
    return votacao[0][0]


# =====================================================================
# PASSO 4: Pipeline de Execução e Avaliação de Performance
# =====================================================================
if __name__ == "__main__":
    try:
        nome_arquivo = "heart_attack_dataset.csv"
        
        
        X_dados, y_dados = carregar_dados_csv("C:/Users/leona/Downloads/heart_attack_dataset.csv")
        print(f"-> Dataset carregado! Total de registros: {len(X_dados)}")
        
        
        X_dados = normalizar_dados(X_dados)
        print("-> Dados escalonados com sucesso (Min-Max entre 0 e 1)!")
        
        
        random.seed(42) 
        dados_combinados = list(zip(X_dados, y_dados))
        random.shuffle(dados_combinados)
        X_dados, y_dados = zip(*dados_combinados)
        
        
        porcentagem_treino = 0.8
        limite = int(len(X_dados) * porcentagem_treino)
        
        X_treino, X_teste = X_dados[:limite], X_dados[limite:]
        y_treino, y_teste = y_dados[:limite], y_dados[limite:]
        
        K_definido = 5               
        metrica_escolhida = 'manhattan' 
        # =================================================================
        
        previsoes_corretas = 0
        print(f"\nAvaliando o KNN com K={K_definido} e Métrica='{metrica_escolhida}'...")
        
        
        for i in range(len(X_teste)):
            predicao = classificar_knn(
                X_treino, 
                y_treino, 
                X_teste[i], 
                k=K_definido, 
                metrica=metrica_escolhida
            )
            
            if predicao == y_teste[i]:
                previsoes_corretas += 1
                
        # Calcula a acurácia final
        acuracia = (previsoes_corretas / len(X_teste)) * 100
        
        print("\n================ RELATÓRIO DE PERFORMANCE ================")
        print(f"Métrica de Distância Utilizada : {metrica_escolhida.upper()}")
        print(f"Parâmetro K Selecionado        : {K_definido}")
        print(f"Total de Testes Realizados     : {len(X_teste)} pacientes")
        print(f"Total de Acertos Confirmados   : {previsoes_corretas}")
        print(f"Acurácia Final do Modelo       : {acuracia:.2f}%")
        print("==========================================================")
        
    except FileNotFoundError:
        print(f"[ERRO] O arquivo '{nome_arquivo}' não foi encontrado.")

-> Dataset carregado! Total de registros: 303
-> Dados escalonados com sucesso (Min-Max entre 0 e 1)!

Avaliando o KNN com K=5 e Métrica='manhattan'...

================ RELATÓRIO DE PERFORMANCE ================
Métrica de Distância Utilizada : MANHATTAN
Parâmetro K Selecionado        : 5
Total de Testes Realizados     : 61 pacientes
Total de Acertos Confirmados   : 51
Acurácia Final do Modelo       : 83.61%
